# ChaoSSM — Phase 2e: Fine-Tuning Fragility Under the Retuned Recipe (Final Diagnostic)

## Why this phase, and why it's the last one

§9.4's risk register named one remaining unverified item: §5.5 found a short, aggressive
fine-tuning burst (50 steps at 10x lr) nearly erased chaossm_free's entire copy-task advantage
at pilot scale — but that test used the *original* hyperparameters (lr=3e-4, γ_init=1.0). §9.2's
retuning diagnostic (lr=1e-4, γ_init=0.5) fixed a *different* problem — base-training collapse
at N=96/192 — under a fine-tuning burst that was never re-applied. This phase closes that gap:
does the recipe that fixed base-training stability *also* fix fine-tuning stability, or are
these two distinct failure modes that both happen to involve the same γ/lr settings?

This is the final notebook in this diagnostic arc. Whatever the result, it closes out the open
items in §9 — no further follow-up phase is planned after this one.

## Design

Direct combination of Phase 2b's fine-tuning-burst protocol and Phase 2d's retuned recipe,
at the two sizes where retuning was validated (N=96, N=192): train each (arm, hyperparameter
configuration, size, seed) combination on copy, then apply the identical burst (50 steps at
10x *that condition's own* training lr — i.e. 10x3e-4 for baseline, 10x1e-4 for retuned) and
measure the accuracy drop. `linear` serves as an unswept reference at each size, as in Phase 2c
and 2d.

3 arms x up to 2 hp-configs (baseline, retuned — only for chaotic arms) x 2 sizes x 4 seeds.

## Setup

In [1]:
import os, json, math, time, warnings, copy as pycopy
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import wilcoxon
from torch.func import jacrev

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

def setup_results_dir(project="ChaoSSM", phase="phase2e"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base = os.path.abspath(f"./{project}_results/{phase}")
        print(f"[info] Not in Colab ({type(e).__name__}); using local dir.")
    os.makedirs(base, exist_ok=True); print("results ->", base); return base

RESULTS = setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

# Reference points from prior phases, for context in the final verdict
PHASE2B_N48_DROP = dict(linear=0.006, chaossm_free=0.076, chaossm_corridor=0.098)
print("Phase 2b (N=48, baseline hyperparameters) fine-tune drop:", PHASE2B_N48_DROP)

torch 2.11.0+cu128 | device: cuda
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase2e
Phase 2b (N=48, baseline hyperparameters) fine-tune drop: {'linear': 0.006, 'chaossm_free': 0.076, 'chaossm_corridor': 0.098}


## Configuration

In [2]:
SMOKE_TEST = False   # final run: chaotic arms x {baseline, retuned} x 2 sizes x 4 seeds + linear reference

@dataclass
class Config:
    state_dim: int = 96
    d_model: int = 96
    vocab: int = 16
    steps: int = 1200
    batch: int = 64
    lr: float = 3e-4
    bptt_k: int = 64
    grad_clip: float = 1.0
    gamma_init: float = 1.0
    warmup_frac: float = 0.10
    dual_lr: float = 0.05
    alpha_max: float = 50.0
    corridor_target: float = 0.15
    corridor_half_width: float = 0.10
    lyap_window: int = 4
    lyap_subsample: int = 3
    lyap_every: int = 20
    copy_len: int = 12
    eval_batch: int = 256
    finetune_steps: int = 50
    finetune_lr_mult: float = 10.0
    sizes: tuple = (96, 192)
    chaotic_arms: tuple = ("chaossm_free", "chaossm_corridor")
    hp_configs: tuple = (
        ("baseline", 3e-4, 1.0),
        ("retuned",  1e-4, 0.5),
    )
    seeds: int = 4

CFG = Config()
if SMOKE_TEST:
    CFG.steps = 150; CFG.seeds = 2; CFG.eval_batch = 64; CFG.finetune_steps = 10
    print("*** SMOKE TEST — pipeline check only ***")
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

{
  "state_dim": "96",
  "d_model": "96",
  "vocab": "16",
  "steps": "1200",
  "batch": "64",
  "lr": "0.0003",
  "bptt_k": "64",
  "grad_clip": "1.0",
  "gamma_init": "1.0",
  "warmup_frac": "0.1",
  "dual_lr": "0.05",
  "alpha_max": "50.0",
  "corridor_target": "0.15",
  "corridor_half_width": "0.1",
  "lyap_window": "4",
  "lyap_subsample": "3",
  "lyap_every": "20",
  "copy_len": "12",
  "eval_batch": "256",
  "finetune_steps": "50",
  "finetune_lr_mult": "10.0",
  "sizes": "(96, 192)",
  "chaotic_arms": "('chaossm_free', 'chaossm_corridor')",
  "hp_configs": "(('baseline', 0.0003, 1.0), ('retuned', 0.0001, 0.5))",
  "seeds": "4"
}


## 1. Model — unchanged, including the fp16-dtype fix carried from Phase 2b onward

In [3]:
def triangle_fold(x):
    return 1.0 - torch.abs(torch.remainder(x, 2.0) - 1.0)

class LinearMap(nn.Module):
    def __init__(self, N):
        super().__init__()
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_lin = nn.Parameter(W * 0.5)
    def forward(self, h):
        return torch.tanh(h @ self.W_lin.T)

class ChaoticMap(nn.Module):
    def __init__(self, N, gamma_init=1.0):
        super().__init__()
        self.w_r = nn.Parameter(torch.zeros(N))
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
        inv = math.log(math.expm1(gamma_init))
        self.w_gamma = nn.Parameter(torch.tensor(float(inv)))
    def get_r(self): return 4.0 * torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self, h):
        r = self.get_r()
        u = h @ self.W_mix.T + r * h * (1 - h)
        return triangle_fold(self.get_gamma() * u)

class ChaoSSMCell(nn.Module):
    def __init__(self, cfg, phi_kind):
        super().__init__()
        self.N = cfg.state_dim
        self.phi_kind = phi_kind
        self.phi = LinearMap(cfg.state_dim) if phi_kind == "linear" \
                   else ChaoticMap(cfg.state_dim, cfg.gamma_init)
        self.A = nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9, 1.1)))
        self.B = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.W_g = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.b_g = nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self, h, x):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        raw = g * self.phi(h @ self.A.T) + (1 - g) * h + x @ self.B.T
        return triangle_fold(raw) if self.phi_kind != "linear" else torch.tanh(raw)
    def init_state(self, b, dev):
        dtype = self.A.dtype
        return torch.zeros(b, self.N, device=dev, dtype=dtype) if self.phi_kind == "linear" \
               else torch.full((b, self.N), 0.5, device=dev, dtype=dtype)

class ChaoSSMLM(nn.Module):
    def __init__(self, cfg, phi_kind, vocab_size, readout_size):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(vocab_size, cfg.d_model)
        self.cell = ChaoSSMCell(cfg, phi_kind)
        self.readout = nn.Linear(cfg.state_dim, readout_size)
    def forward(self, tok, return_traj=False):
        B, T = tok.shape
        x = self.emb(tok); h = self.cell.init_state(B, tok.device)
        outs, traj = [], []
        for t in range(T):
            h = self.cell.step(h, x[:, t]); outs.append(self.readout(h))
            if return_traj: traj.append(h)
            if (t + 1) % self.cfg.bptt_k == 0: h = h.detach()
        lg = torch.stack(outs, 1)
        return (lg, torch.stack(traj, 1), x) if return_traj else lg

## 2. Estimator, task, accuracy — unchanged

In [4]:
def _full_jac_fn(model, x):
    return lambda z: model.cell.step(z.unsqueeze(0), x.unsqueeze(0))[0]

def lambda_window(model, traj, xseq, W, n_sample):
    B, T, N = traj.shape
    if T <= W + 1:
        return torch.zeros((), device=traj.device)
    lams = []
    for b in torch.randperm(B)[:n_sample]:
        t0 = int(torch.randint(0, T - W - 1, (1,)))
        M = torch.eye(N, device=traj.device, dtype=traj.dtype)
        for k in range(W):
            M = jacrev(_full_jac_fn(model, xseq[b, t0+k]))(traj[b, t0+k]) @ M
        lams.append(torch.log(torch.linalg.matrix_norm(M, ord=2) + 1e-12) / W)
    return torch.stack(lams).mean()

def make_copy_batch(cfg, batch, device, gen):
    L, V = cfg.copy_len, cfg.vocab
    sep, pad = V, V + 1
    src = torch.randint(0, V, (batch, L), device=device, generator=gen)
    inp = torch.cat([src, torch.full((batch,1), sep, device=device),
                     torch.full((batch,L), pad, device=device)], 1)
    tgt = torch.cat([torch.full((batch,L+1), -100, device=device), src], 1)
    return inp, tgt

VOCAB_SIZE = CFG.vocab + 2
CHANCE = 1.0 / CFG.vocab

def copy_accuracy(logits, tgt):
    mask = tgt != -100
    pred = logits.argmax(-1)
    return ((pred == tgt) & mask).float().sum().item() / mask.float().sum().clamp(min=1).item()

## 3. Trainer — `lr`/`gamma_init` vary by hp_config, as in Phase 2d

In [5]:
def train_one(arm, size, lr, gamma_init, seed, cfg):
    cfg = pycopy.deepcopy(cfg)
    cfg.state_dim = size; cfg.d_model = size; cfg.lr = lr; cfg.gamma_init = gamma_init
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    phi_kind = "linear" if arm == "linear" else "chaossm"
    model = ChaoSSMLM(cfg, phi_kind, VOCAB_SIZE, VOCAB_SIZE).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.95))

    use_lyap = (arm == "chaossm_corridor")
    lo = hi = None
    if use_lyap:
        lo = cfg.corridor_target - cfg.corridor_half_width
        hi = cfg.corridor_target + cfg.corridor_half_width
    a_hi = a_lo = 0.0
    warmup_steps = int(cfg.warmup_frac * cfg.steps)
    diverged = False
    final_acc = float("nan")

    for step in range(cfg.steps):
        inp, tgt = make_copy_batch(cfg, cfg.batch, DEVICE, gen)
        need_traj = (arm != "linear")
        out = model(inp, return_traj=need_traj)
        logits, traj, xseq = out if need_traj else (out, None, None)
        L_task = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1),
                                 ignore_index=-100)

        L_pen = torch.zeros((), device=DEVICE)
        if arm != "linear":
            in_warmup = step < warmup_steps
            lam = lambda_window(model, traj, xseq, cfg.lyap_window, cfg.lyap_subsample)
            lam_val = float(lam.detach())
            if use_lyap and not in_warmup:
                L_pen = a_hi * F.relu(lam - hi) + a_lo * F.relu(lo - lam)

        opt.zero_grad(set_to_none=True); (L_task + L_pen).backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
        opt.step()

        if use_lyap and step >= warmup_steps:
            a_hi = float(np.clip(a_hi + cfg.dual_lr * (lam_val - hi), 0.0, cfg.alpha_max))
            a_lo = float(np.clip(a_lo + cfg.dual_lr * (lo - lam_val), 0.0, cfg.alpha_max))

        if not (math.isfinite(L_task.item()) and math.isfinite(gn)):
            diverged = True; break

        if step % 20 == 0:
            with torch.no_grad():
                final_acc = copy_accuracy(logits, tgt)

    return model, cfg, dict(arm=arm, state_dim=size, hp_lr=lr, hp_gamma_init=gamma_init,
                            seed=seed, diverged=diverged, final_acc=final_acc)

## 4. Fine-tuning-stability test — identical protocol to Phase 2b/2c

In [6]:
def eval_finetune_stability(model, run_cfg, arm, size, hp_name, seed):
    model = pycopy.deepcopy(model)
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed + 30_000)

    with torch.no_grad():
        inp0, tgt0 = make_copy_batch(run_cfg, run_cfg.eval_batch, DEVICE, gen)
        acc_before = copy_accuracy(model(inp0), tgt0)

    opt = torch.optim.AdamW(model.parameters(), lr=run_cfg.lr * run_cfg.finetune_lr_mult, betas=(0.9, 0.95))
    for _ in range(run_cfg.finetune_steps):
        inp, tgt = make_copy_batch(run_cfg, run_cfg.batch, DEVICE, gen)
        logits = model(inp)
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1), ignore_index=-100)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), run_cfg.grad_clip)
        opt.step()

    with torch.no_grad():
        inp1, tgt1 = make_copy_batch(run_cfg, run_cfg.eval_batch, DEVICE, gen)
        acc_after = copy_accuracy(model(inp1), tgt1)

    return dict(arm=arm, state_dim=size, hp_config=hp_name, seed=seed,
               acc_before=acc_before, acc_after=acc_after, acc_drop=acc_before - acc_after)

## 5. Run — chaotic arms x {baseline, retuned} x 2 sizes x 4 seeds, plus linear reference

In [7]:
ft_rows = []
total = (len(CFG.chaotic_arms) * len(CFG.hp_configs) * len(CFG.sizes) * CFG.seeds
        + len(CFG.sizes) * CFG.seeds)
done = 0

for size in CFG.sizes:
    for hp_name, lr, gamma_init in CFG.hp_configs:
        for arm in CFG.chaotic_arms:
            for sd in range(CFG.seeds):
                model, run_cfg, s = train_one(arm, size, lr, gamma_init, sd, CFG)
                ft = eval_finetune_stability(model, run_cfg, arm, size, hp_name, sd)
                ft_rows.append(ft); done += 1
                flag = " DIVERGED" if s["diverged"] else ""
                print(f"[{done}/{total}] N={size:4d} {hp_name:9s} {arm:18s} seed={sd}  "
                      f"before={ft['acc_before']:.4f}  after={ft['acc_after']:.4f}  "
                      f"drop={ft['acc_drop']:+.4f}{flag}")

for size in CFG.sizes:
    for sd in range(CFG.seeds):
        model, run_cfg, s = train_one("linear", size, 3e-4, 1.0, sd, CFG)
        ft = eval_finetune_stability(model, run_cfg, "linear", size, "reference", sd)
        ft_rows.append(ft); done += 1
        print(f"[{done}/{total}] N={size:4d} {'reference':9s} {'linear':18s} seed={sd}  "
              f"before={ft['acc_before']:.4f}  after={ft['acc_after']:.4f}  drop={ft['acc_drop']:+.4f}")

FT = pd.DataFrame(ft_rows)
savetable(FT, "phase2e_finetune.csv")

[1/40] N=  96 baseline  chaossm_free       seed=0  before=0.0632  after=0.0570  drop=+0.0062
[2/40] N=  96 baseline  chaossm_free       seed=1  before=0.0615  after=0.0658  drop=-0.0042
[3/40] N=  96 baseline  chaossm_free       seed=2  before=0.0680  after=0.0648  drop=+0.0033
[4/40] N=  96 baseline  chaossm_free       seed=3  before=0.0550  after=0.0713  drop=-0.0163
[5/40] N=  96 baseline  chaossm_corridor   seed=0  before=0.2064  after=0.0664  drop=+0.1400
[6/40] N=  96 baseline  chaossm_corridor   seed=1  before=0.2402  after=0.0573  drop=+0.1829
[7/40] N=  96 baseline  chaossm_corridor   seed=2  before=0.2243  after=0.0605  drop=+0.1637
[8/40] N=  96 baseline  chaossm_corridor   seed=3  before=0.0674  after=0.0632  drop=+0.0042
[9/40] N=  96 retuned   chaossm_free       seed=0  before=0.1807  after=0.1224  drop=+0.0583
[10/40] N=  96 retuned   chaossm_free       seed=1  before=0.1882  after=0.1729  drop=+0.0153
[11/40] N=  96 retuned   chaossm_free       seed=2  before=0.1758  af

## 6. Does retuning fix fine-tuning stability too, or is it a separate problem?

In [8]:
summary = FT.groupby(["state_dim","hp_config","arm"])[["acc_before","acc_after","acc_drop"]].agg(
    ["mean","std"]).round(4)
print(summary.to_string())
savetable(summary.reset_index(), "phase2e_summary.csv")

def paired(size, arm, col="acc_drop"):
    x = FT[(FT.state_dim==size)&(FT.hp_config=="retuned")&(FT.arm==arm)].sort_values("seed")[col].values
    y = FT[(FT.state_dim==size)&(FT.hp_config=="baseline")&(FT.arm==arm)].sort_values("seed")[col].values
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    if len(x) < 3 or np.allclose(x, y):
        return dict(state_dim=size, arm=arm, n=len(x), p=np.nan,
                    median_diff=float(np.median(x-y)) if len(x) else np.nan)
    st, p = wilcoxon(x, y)
    return dict(state_dim=size, arm=arm, n=len(x), p=float(p), median_diff=float(np.median(x-y)))

ST = pd.DataFrame([paired(sz, arm) for sz in CFG.sizes for arm in CFG.chaotic_arms])
print()
print("retuned vs. baseline, acc_drop (negative median_diff = retuning REDUCES the drop = more stable):")
print(ST.to_string(index=False))
savetable(ST, "phase2e_stats.csv")
print(f"\nmin achievable two-sided p at n={CFG.seeds}: {2/(2**CFG.seeds):.4f}")

                                     acc_before         acc_after         acc_drop        
                                           mean     std      mean     std     mean     std
state_dim hp_config arm                                                                   
96        baseline  chaossm_corridor     0.1846  0.0793    0.0618  0.0039   0.1227  0.0809
                    chaossm_free         0.0619  0.0054    0.0647  0.0059  -0.0028  0.0100
          reference linear               0.0786  0.0286    0.0683  0.0058   0.0103  0.0265
          retuned   chaossm_corridor     0.1786  0.0061    0.1681  0.0094   0.0105  0.0094
                    chaossm_free         0.1822  0.0053    0.1395  0.0446   0.0427  0.0459
192       baseline  chaossm_corridor     0.0615  0.0033    0.0633  0.0023  -0.0018  0.0022
                    chaossm_free         0.0603  0.0052    0.0621  0.0033  -0.0018  0.0079
          reference linear               0.0693  0.0056    0.0640  0.0063   0.0054  0.0106

In [9]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for i, size in enumerate(CFG.sizes):
    d = FT[FT.state_dim == size]
    labels, means, stds = [], [], []
    for hp in ["baseline", "retuned"]:
        for arm in CFG.chaotic_arms:
            dd = d[(d.hp_config==hp)&(d.arm==arm)]
            labels.append(f"{hp}\n{arm.split('_')[1]}"); means.append(dd.acc_drop.mean()); stds.append(dd.acc_drop.std())
    lin = d[d.arm=="linear"]
    labels.append("linear\n(ref)"); means.append(lin.acc_drop.mean()); stds.append(lin.acc_drop.std())
    colors = ["tab:orange","tab:orange","tab:green","tab:green","gray"]
    ax[i].bar(labels, means, yerr=stds, color=colors, alpha=.85)
    ax[i].axhline(PHASE2B_N48_DROP["chaossm_free"], color="purple", ls="--", label="N=48 pilot-scale drop (chaossm_free)")
    ax[i].set(title=f"N={size}", ylabel="fine-tune-induced accuracy drop" if i==0 else "")
    ax[i].tick_params(axis="x", labelsize=7); ax[i].legend(fontsize=7)
fig.tight_layout(); savefig(fig, "phase2e_finetune_recovery.png"); plt.close(fig)

saved /content/drive/My Drive/ChaoSSM/results/phase2e/phase2e_finetune_recovery.png


## 7. Final verdict

| Observation | Reading |
|---|---|
| `retuned` shows significantly smaller `acc_drop` than `baseline`, at both sizes | The single retuning fix (§9.2) resolves both problems it was tested against — base-training collapse and fine-tuning fragility share a common cause. Clean, simple story for the manuscript: one recipe change, two problems solved. |
| `retuned` drop is smaller but still substantial (e.g. comparable to or larger than linear's) | Retuning helps but doesn't fully close the gap to linear's stability — report as a partial fix, and note the residual fine-tuning cost as a real, quantified limitation to carry into §9.3 regardless of base-training fixes. |
| `retuned` shows no improvement in `acc_drop` over `baseline` | Fine-tuning fragility and base-training collapse are two distinct failure modes with different causes. The §9.2 recipe fixes one but not the other — §9.3 needs a separate mitigation for fine-tuning specifically (e.g. lower fine-tuning learning rates than the 10x-burst tested here). |

This closes the diagnostic arc that began with §5.5's robustness phase. Whatever the outcome,
report it in §5.5/§9 exactly as found — this phase was designed to give a definitive answer
either way, not to confirm a hoped-for one.

In [10]:
def flat_dict(series_groupby):
    d = series_groupby.agg(["mean", "std"]).round(4)
    return {f"{k[0]}|{k[1]}|{k[2]}": v for k, v in d.to_dict("index").items()}

verdict = dict(
    smoke_test=SMOKE_TEST,
    phase2b_n48_reference=PHASE2B_N48_DROP,
    finetune_drop_summary=flat_dict(FT.groupby(["state_dim","hp_config","arm"]).acc_drop),
    stats=ST.to_dict("records"),
    retuning_fixes_finetune_fragility=bool((ST.median_diff < 0).all() and (ST.p <= 0.125).all()),
    config=dict(steps=CFG.steps, seeds=CFG.seeds, sizes=CFG.sizes,
                finetune_steps=CFG.finetune_steps, finetune_lr_mult=CFG.finetune_lr_mult),
)
savejson(verdict, "phase2e_verdict.json")
print(json.dumps(verdict, indent=2, default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase2e/phase2e_verdict.json
{
  "smoke_test": false,
  "phase2b_n48_reference": {
    "linear": 0.006,
    "chaossm_free": 0.076,
    "chaossm_corridor": 0.098
  },
  "finetune_drop_summary": {
    "96|baseline|chaossm_corridor": {
      "mean": 0.1227,
      "std": 0.0809
    },
    "96|baseline|chaossm_free": {
      "mean": -0.0028,
      "std": 0.01
    },
    "96|reference|linear": {
      "mean": 0.0103,
      "std": 0.0265
    },
    "96|retuned|chaossm_corridor": {
      "mean": 0.0105,
      "std": 0.0094
    },
    "96|retuned|chaossm_free": {
      "mean": 0.0427,
      "std": 0.0459
    },
    "192|baseline|chaossm_corridor": {
      "mean": -0.0018,
      "std": 0.0022
    },
    "192|baseline|chaossm_free": {
      "mean": -0.0018,
      "std": 0.0079
    },
    "192|reference|linear": {
      "mean": 0.0054,
      "std": 0.0106
    },
    "192|retuned|chaossm_corridor": {
      "mean": 0.1503,
      "std": 0.059
    },
    "